# Libraries and formatting

In [88]:
import math

def formatStates(states):
    print("    x     |     y")
    for state in states:
        x, y = state
        print(str(x)[:7] + " "*(10-len(str(x)[:7])) + "| " + str(y)[:7] + " "*(10-len(str(y)[:7])))

# Fibonacci
## Iterative

In [89]:
def fibonacciIterative():
    # Starting digits
    a = 1#int(input("What do you want the first Fibonacci term to be?: "))
    b = 1#int(input("What do you want the second Fibonacci term to be?: "))
    sequence = [a, b]

    # How many terms?
    terms = 20#int(input("How many terms of the Fibonacci Sequence do you want? (Minimum 2): "))

    # Iteration
    for _ in range(0,terms-2):
        # print(sequence)
        sequence.append(sequence[-2] + sequence[-1])

    print(sequence)

    if sequence[-2] != 0:
        print(f"φ ≈ {sequence[-1] / sequence[-2]}")
        
fibonacciIterative()

[1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765]
φ ≈ 1.6180339631667064


## Recursive

In [90]:
def fibonacciRecursive():
    # Starting digits
    a = 1#int(input("What do you want the first Fibonacci term to be?: "))
    b = 1#int(input("What do you want the second Fibonacci term to be?: "))
    sequence = [a, b]

    # How many terms?
    terms = 20#int(input("How many terms of the Fibonacci Sequence do you want? (Minimum 2): "))

    # Recursion
    def step(sequence):
        return sequence[-2] + sequence[-1]

    for _ in range(0, terms - 2):
        #print(sequence)
        sequence.append(step(sequence))

    print(sequence)

    if sequence[-2] != 0:
        print(f"φ ≈ {sequence[-1] / sequence[-2]}")


fibonacciRecursive()

[1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765]
φ ≈ 1.6180339631667064


# Apply to differential equations numerical methods
## Apply step function to 1st order ODE in terms of x only
### Idea: 
$$\frac{dy}{dx} \approx \frac{y_1 - y_0}{h} \implies y_1 \approx y_0 + h (\frac{dy}{dx})_0$$
##### Try: 
$$\frac{dx}{dt} = kx ,\, k \in \mathbb{Z}^+\\
x_0 = 1, k = -1$$

In [91]:
def dx_dt_equals_kx(k):
    # Starting state stored as tuple of (x, t)
    states = [(1, 0)]

    # How many terms?
    terms = 20 #int(input("How many steps do you want?: "))
    h = 0.1 #float(input("What step size do you want?: "))

    def step(x, t):
        return (x + h*k*x, t + h)

    for _ in range(0, terms - 1):
        x, t = states[-1]
        states.append(step(x, t))

    return states

states = dx_dt_equals_kx(-1)
formatStates(states)

    x     |     y
1         | 0         
0.9       | 0.1       
0.81      | 0.2       
0.72900   | 0.30000   
0.65610   | 0.4       
0.59049   | 0.5       
0.53144   | 0.6       
0.47829   | 0.7       
0.43046   | 0.79999   
0.38742   | 0.89999   
0.34867   | 0.99999   
0.31381   | 1.09999   
0.28242   | 1.2       
0.25418   | 1.3       
0.22876   | 1.40000   
0.20589   | 1.50000   
0.18530   | 1.60000   
0.16677   | 1.70000   
0.15009   | 1.80000   
0.13508   | 1.90000   


# Applying to a general derivative function
## (Forward) Euler Method
This can then be expanded for any $\frac{dy}{dx}$ known as the (forward) Euler Method.
$$\frac{dy}{dx} \approx \frac{y_1 - y_0}{h} \implies y_1 \approx y_0 + h (\frac{dy}{dx})_0$$

In [92]:
# This can be changed to any function desired and will be called in future numerical method implementations
def derivative(x, y):
    return y - x**2 + 1

def initializeStates():
    x0 = 0
    y0 = 0.5
    h = 0.1
    states = [(x0, y0)]
    steps = 10
    return x0, y0, h, states, steps

In [93]:
x0, y0, h, states, steps = initializeStates()

# Applies one step of the Euler method: approximating the gradient of the function on the interval 'h' using the derivative function
def eulerStep(h, x, y):
    return (x + h, y + h*derivative(x,y))

def eulersMethod(states, steps, h,derivative, x0, y0,):
    for _ in range(0, steps):
        x0, y0 = eulerStep(h, x0, y0)
        states.append((x0,y0))
    return states
    


states = eulersMethod(states, steps, h, derivative, x0, y0)
formatStates(states)


    x     |     y
0         | 0.5       
0.1       | 0.65      
0.2       | 0.81400   
0.30000   | 0.99140   
0.4       | 1.18154   
0.5       | 1.38369   
0.6       | 1.59706   
0.7       | 1.82076   
0.79999   | 2.05384   
0.89999   | 2.29523   
0.99999   | 2.54375   


# Evolve to form the Leapfrog method
Instead of taking the slope from the initial position but by using the previous point and the current gradient to estimate the next step. This is often started by doing one step of Euler's method first.
$\\ y_{2} = y_{0} + 2h(\frac{dy}{dx})_1$

In [94]:
x0, y0, h, states, steps = initializeStates()

# Firstly apply one step of Euler's
x1, y1 = eulerStep(h, x0, y0)
states.append((x1,y1))

def leapfrogStep(h, x0, y0, x1, y1):
    # Apply leapfrog method
    x2 = x1 + h
    y2 = y0 + 2*h*derivative(x1,y1)
    return x1, y1, x2, y2

def leapfrogMethod(states, steps, h, x0, y0, x1, y1):
    for _ in range(0,steps - 1):
        x0, y0, x1, y1 = leapfrogStep(h, x0, y0, x1, y1)
        states.append((x1,y1))
    return states
    
states = leapfrogMethod(states, steps, h, x0, y0, x1, y1)
formatStates(states)

    x     |     y
0         | 0.5       
0.1       | 0.65      
0.2       | 0.82800   
0.30000   | 1.0076    
0.4       | 1.21152   
0.5       | 1.41790   
0.6       | 1.64510   
0.7       | 1.87492   
0.79999   | 2.12208   
0.89999   | 2.37134   
0.99999   | 2.63435   


## Or evolve to the Midpoint method (RK2)
For $\frac{dy}{dx} = f(x,y)$ <br>
Starting from Euler's method: $y(x+h) \approx y(x) + hf(x,y)$ <br>
We assume the gradient at the midpoint of the interval is a better approximation for the average gradient throughout the interval than the gradient at the start point. <br>
$\implies y(x+h) \approx y(x) + hf(x+\frac{h}{2},\, y(x+\frac{h}{2}))$ <br>
But $y(x+\frac{h}{2})$ cannot be directly evaluated as the $y$ value is not known. <br>
So an Euler's method half-step is used to approximate this point: $y(x+\frac{h}{2}) \approx y(x) + \frac{h}{2} f(x,y)$ <br>
$\implies y(x+h) \approx y(x) + hf\left(x+\frac{h}{2},\, y+\frac{h}{2}f(x,y)\right)$

In [95]:
x0, y0, h, states, steps = initializeStates()

def midpointStep(h, x0, y0):
    # Apply midpoint / RK2 method
    x1 = x0 + h
    xmid = x0 + h/2
    ymid = y0 + h/2 * derivative(x0, y0)
    y1 = y0 + h*derivative(xmid, ymid)
    return (x1, y1)

def midpointMethod(states, steps, h, x0, y0):
    for _ in range(0, steps):
        x0, y0 = midpointStep(h, x0, y0)
        states.append((x0,y0))
    return states

states = midpointMethod(states, steps, h, x0, y0)
formatStates(states)


    x     |     y
0         | 0.5       
0.1       | 0.65725   
0.2       | 0.82896   
0.30000   | 1.01455   
0.4       | 1.21338   
0.5       | 1.42473   
0.6       | 1.64783   
0.7       | 1.88180   
0.79999   | 2.12569   
0.89999   | 2.37844   
0.99999   | 2.63887   


## The Runge-Kutta Method (RK4)
Euler's method uses a single slope to approximate so is only accurate if the slope does not change much over the interval. <br>
The Midpoint method uses two slopes so is more accurate; RK4 uses four slopes. <br>
For $\frac{dy}{dx} = f(x,y)$
## Slope evaluations
$k_1 = f(x,y)$ | Slope at the start of the interval <br>
$k_2 = f(x+\frac{h}{2},\, y+\frac{h}{2}k_1)$ | Slope estimate as evaluated in the midpoint method (Euler half-step) <br>
$k_3 = f(x+\frac{h}{2},\, y+\frac{h}{2}k_2)$ | An improved midpoint slope using the slope at the Euler half-step <br>
$k_4 = f(x+h,\, y+hk_3$ | Slope estimate for the end of the interval <br>
These are then weighted to match the Taylor expansion giving:
$$y(x+h) \approx y(x) + \frac{h}{6}\left(k_1 + 2k_2 + 2k_3 + k_4\right)$$

In [ ]:
x0, y0, h, states, steps = initializeStates()

def rungeKuttaStep(h, x0, y0):
    # Apply Runge-Kutta / RK4 method
    k1 = derivative(x0, y0)
    k2 = derivative(x0 + h/2, y0 + h/2 * k1)
    k3 = derivative(x0 + h/2, y0 + h/2 * k2)
    k4 = derivative(x0 + h, y0 + h * k3)
    x1 = x0 + h
    y1 = y0 + h/6 * (k1 + 2*k2 + 2*k3 + k4)
    return (x1, y1)

def rungeKuttaMethod(states, steps, h, x0, y0):
    for _ in range(0, steps):
        x0, y0 = rungeKuttaStep(h, x0, y0)
        states.append((x0,y0))
    return states

states = rungeKuttaMethod(states, steps, h, x0, y0)
formatStates(states)

    x     |     y
0         | 0.5       
0.1       | 0.65741   
0.2       | 0.82929   
0.30000   | 1.01507   
0.4       | 1.21408   
0.5       | 1.42563   
0.6       | 1.64893   
0.7       | 1.88312   
0.79999   | 2.12722   
0.89999   | 2.38019   
0.99999   | 2.64085   
